# CropCheckUp — BiRefNet background removal

Removes the background from every PlantVillage image in `rasagyavatsal/cropcheckup-dataset` with
**BiRefNet-General** (`ZhengPeng7/BiRefNet`) and saves transparent PNGs.

Output (in `/kaggle/working`):
- `segmented__<class>.zip` — one per class, each containing `segmented/<class>/<name>.png` (RGBA)
- `manifest.json` — classes completed, image counts, failures, fallbacks, throughput
- `preview.png` — a few results for a quick visual check

Classes are zipped one at a time so a run that hits the time budget still leaves complete classes behind.

BiRefNet finds no subject in close-ups where the leaf fills the frame (common for corn) and returns an empty or
lesion-only mask. Masks covering less than `MIN_COVERAGE` fall back to the full original image, fully opaque;
those files are listed under `fallback_full_image` in the manifest.

In [ ]:
import subprocess, sys

# BiRefNet's remote code doesn't load under transformers 5.x.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40,<5", "timm", "kornia", "einops"])

In [ ]:
from pathlib import Path

# Kaggle's mount layout under /kaggle/input has changed over time, so locate raw/ instead of hard-coding it.
RAW_DIR = next(p for p in Path("/kaggle/input").rglob("raw") if p.parent.name == "plantvillage")
WORK_DIR = Path("/kaggle/working")
MODEL_ID = "ZhengPeng7/BiRefNet"
IMG_SIZE = 1024
BATCH_SIZE = 16  # split across GPUs when there are two
MIN_COVERAGE = 0.10  # below this foreground fraction, keep the whole image instead of the mask
CLASSES = None  # class folder names to process; None = all
LIMIT_PER_CLASS = None  # cap images per class (timing runs); None = all
TIME_BUDGET_H = 10  # don't start a new class after this; the largest class takes ~45 min, Kaggle's limit is 12h
print(RAW_DIR)

In [ ]:
import time
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
torch.set_float32_matmul_precision("high")

model = AutoModelForImageSegmentation.from_pretrained(MODEL_ID, trust_remote_code=True)
model.to(device, dtype).eval()
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
print("device:", device, torch.cuda.device_count(), torch.cuda.get_device_name() if device == "cuda" else "")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
to_pil = transforms.ToPILImage()


class Images(torch.utils.data.Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return transform(Image.open(self.paths[i]).convert("RGB")), i

In [ ]:
import json, shutil, zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}


def save_cutout(path, pred, out_dir):
    # Returns True when the mask was too small and the full image was kept instead.
    image = Image.open(path).convert("RGB")
    fallback = (pred > 0.5).float().mean().item() < MIN_COVERAGE
    mask = Image.new("L", image.size, 255) if fallback else to_pil(pred).resize(image.size, Image.BILINEAR)
    image.putalpha(mask)
    # Zero the colour under fully transparent pixels so the background can't resurface via .convert("RGB").
    rgba = np.array(image)
    rgba[rgba[..., 3] == 0, :3] = 0
    Image.fromarray(rgba).save(out_dir / f"{path.stem}.png", optimize=True)
    return fallback


classes = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())
if CLASSES:
    classes = [c for c in classes if c in CLASSES]

manifest = {"completed": {}, "skipped_for_time": [], "failures": [], "fallback_full_image": []}
start = time.time()
n_done = 0
saver = ThreadPoolExecutor(4)
for ci, cls in enumerate(classes):
    if (time.time() - start) / 3600 > TIME_BUDGET_H:
        manifest["skipped_for_time"] = classes[ci:]
        print("time budget reached; not started:", classes[ci:])
        break

    paths = sorted(p for p in (RAW_DIR / cls).iterdir() if p.suffix.lower() in IMAGE_EXTS)[:LIMIT_PER_CLASS]
    out_dir = WORK_DIR / "segmented" / cls
    out_dir.mkdir(parents=True, exist_ok=True)
    loader = torch.utils.data.DataLoader(Images(paths), batch_size=BATCH_SIZE, num_workers=4)

    pending = []
    for x, idx in loader:
        with torch.no_grad():
            preds = model(x.to(device, dtype))[-1].sigmoid().float().cpu()
        pending += [(paths[i], saver.submit(save_cutout, paths[i], pred.squeeze(), out_dir)) for pred, i in zip(preds, idx.tolist())]
    for path, fut in pending:
        try:
            if fut.result():
                manifest["fallback_full_image"].append(f"{cls}/{path.name}")
        except Exception as e:
            manifest["failures"].append({"path": str(path), "error": repr(e)})

    # PNGs are already compressed, so store rather than deflate.
    saved = sorted(out_dir.iterdir())
    with zipfile.ZipFile(WORK_DIR / f"segmented__{cls}.zip", "w", zipfile.ZIP_STORED) as zf:
        for f in saved:
            zf.write(f, f"segmented/{cls}/{f.name}")
    if ci >= 2:  # leave the first two classes unzipped for the preview cell
        shutil.rmtree(out_dir)

    manifest["completed"][cls] = {"input": len(paths), "saved": len(saved)}
    n_done += len(paths)
    elapsed = time.time() - start
    print(f"[{ci + 1}/{len(classes)}] {cls}: {len(saved)}/{len(paths)}  |  {n_done / elapsed:.2f} img/s, {elapsed / 60:.1f} min elapsed", flush=True)

saver.shutdown()
manifest["images_per_second"] = n_done / (time.time() - start)
manifest["elapsed_min"] = (time.time() - start) / 60
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1))
print(json.dumps({k: v for k, v in manifest.items() if k not in ("completed", "fallback_full_image")}, indent=1))
print(len(manifest["fallback_full_image"]), "images kept whole (mask below MIN_COVERAGE)")

In [ ]:
import matplotlib.pyplot as plt

samples = sorted((WORK_DIR / "segmented").rglob("*.png"))[:8]
fig, axes = plt.subplots(2, len(samples), figsize=(2.2 * len(samples), 4.6))
for col, cut in enumerate(samples):
    original = next((RAW_DIR / cut.parent.name).glob(f"{cut.stem}.*"))
    rgba = Image.open(cut)
    on_black = Image.new("RGB", rgba.size, (0, 0, 0))
    on_black.paste(rgba, mask=rgba.split()[-1])
    for row, img in enumerate((Image.open(original), on_black)):
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
    axes[0, col].set_title(cut.parent.name[:22], fontsize=7)
plt.tight_layout()
fig.savefig(WORK_DIR / "preview.png", dpi=90)
shutil.rmtree(WORK_DIR / "segmented")
plt.show()